# Topic Modeling with LDA

## Setup and Imports

In [46]:
import pandas as pd
import numpy as np
from random import seed

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, NMF 

from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA, TruncatedSVD as SVD

In [47]:
random_state=42

In [48]:
sns.set_theme(style="white")
colors = "YlGnBu"

In [49]:
model_type = 'lda' # or 'nmf'
data_home = "../input"


In [50]:
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

In [51]:
OHCO = ['doc_title', 'chunk_id','token_id']
CHUNKS = OHCO[:2]
STORIES = OHCO[:1]

BAG = CHUNKS

In [52]:
LIB = pd.read_csv('data/pg2591-LIB.csv')
LIB.set_index('doc_title', inplace=True)
LIB.head()

,volume,n_tokens,n_paras
doc_title,,,
THE GOLDEN BIRD,1,2815,19
HANS IN LUCK,1,2736,19
JORINDA AND JORINDEL,1,1238,16
THE TRAVELLING MUSICIANS,1,1555,8
OLD SULTAN,1,974,5


In [53]:
TOKENS = pd.read_csv('data/chunked_tokens.csv').set_index(OHCO).dropna()
TOKENS

pos_tuple  pos token_str term_str pos_group
doc_title chunk_id token_id                                                   
ASHPUTTEL 0        0           ('the', 'DT')   DT       the      the        DT
                   1          ('wife', 'NN')   NN      wife     wife        NN
                   2            ('of', 'IN')   IN        of       of        IN
                   3             ('a', 'DT')   DT         a        a        DT
                   4          ('rich', 'JJ')   JJ      rich     rich        JJ
...                                      ...  ...       ...      ...       ...
TOM THUMB 19       66           ('s', 'VBZ')  VBZ         s        s        VB
                   67           ('no', 'DT')   DT        no       no        DT
                   68        ('place', 'NN')   NN     place    place        NN
                   69         ('like', 'IN')   IN      like     like        IN
                   70         ('home', 'NN')   NN      home     home        NN

[115213 rows x 5 columns]

In [54]:
DOCS = TOKENS[TOKENS.pos.str.match(r'^NNS?$')]\
    .groupby(BAG).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

DOCS

doc_str
doc_title chunk_id                                                   
ASHPUTTEL 0         wife man end drew daughter bedside girl i watc...
          1         fair face foul heart sorry time girl goodforno...
          2         hearth ashes course dirty ashputtel father wif...
          3         daughter mother s grave tears tree times day b...
          4         hair shoes sashes king s feast ball mother not...
...                                                               ...
TOM THUMB 15        wolf chat friend i treat s wolf house father s...
          16        content way tom shout noise wolf everybody hou...
          17        wolf woodman axe wife scythe do woodman i head...
          18        ah father world i way home air father i mouseh...
          19        clothes ones journey home father mother peace ...

[784 rows x 1 columns]

## Create Vector Space

In [55]:
from sklearn.feature_extraction import text

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(['yes']))
my_stop_words[:10]
# my_stop_words.append('said') # add more words to stop words because they appeared it most topics and ruined the topics
# my_stop_words.append('came')
# my_stop_words.append('went')

['detail',
 'please',
 'whereupon',
 'his',
 'while',
 'amount',
 'anyhow',
 'somewhere',
 'con',
 'a']

In [56]:
count_engine = CountVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words) # Got some advice from claude to lower min ax max df because corpus ins amll
count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
DTM

account  advice  ah  air  alas  ale  anger  animals  \
doc_title chunk_id                                                        
ASHPUTTEL 0               0       0   0    0     0    0      0        0   
          1               0       0   0    0     0    0      0        0   
          2               0       0   0    0     0    0      0        0   
          3               0       0   0    0     0    0      0        0   
          4               0       0   0    0     0    0      0        0   
...                     ...     ...  ..  ...   ...  ...    ...      ...   
TOM THUMB 15              0       0   0    0     0    0      0        0   
          16              0       0   0    0     0    0      0        0   
          17              0       0   1    0     0    0      0        0   
          18              0       0   1    1     0    0      0        0   
          19              0       0   0    0     0    0      0        0   

                    answer  apple  ...  woods  word  words  work  world  \
doc_title chunk_id                 ...                                    
ASHPUTTEL 0              0      0  ...      0     0      0     0      0   
          1              0      0  ...      0     0      0     1      0   
          2              0      0  ...      0     0      0     0      0   
          3              0      0  ...      0     0      0     0      0   
          4              0      0  ...      0     0      0     0      0   
...                    ...    ...  ...    ...   ...    ...   ...    ...   
TOM THUMB 15             0      0  ...      0     0      0     0      0   
          16             0      0  ...      0     0      0     0      0   
          17             0      0  ...      0     0      0     0      0   
          18             0      0  ...      0     0      0     0      2   
          19             0      0  ...      0     0      0     0      0   

                    wretch  yard  year  years  youth  
doc_title chunk_id                                    
ASHPUTTEL 0              0     0     0      0      0  
          1              0     0     0      0      0  
          2              0     0     0      0      0  
          3              0     0     0      0      0  
          4              0     0     0      0      0  
...                    ...   ...   ...    ...    ...  
TOM THUMB 15             0     0     0      0      0  
          16             0     0     0      0      0  
          17             0     0     0      0      0  
          18             0     0     0      0      0  
          19             0     0     0      0      0  

[784 rows x 707 columns]

In [57]:
# Used claude code to help with tfidf engine and model because I want nmf to get better topics than lda
# tfidf_engine = TfidfVectorizer(max_df=.75, min_df=5, stop_words=my_stop_words)
# tfidf_model = tfidf_engine.fit_transform(DOCS.doc_str)
# TERMS = tfidf_engine.get_feature_names_out()
# TFIDF = tfidf_engine.fit_transform(DOCS.doc_str)
# VOCAB = pd.DataFrame(index=TERMS)
# VOCAB.index.name = 'term_str'
# DTM = pd.DataFrame(tfidf_model.toarray(), index=DOCS.index, columns=TERMS)
# DTM


## Generate Model with 20 Topics

In [58]:
n_topics = 5
max_iter = 100
n_top_terms = 5
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]

In [59]:
if model_type == 'lda':
    topic_engine = LDA(n_components=n_topics, max_iter=max_iter,random_state=random_state)
elif model_type == 'nmf':
    topic_engine = NMF(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(count_model)

## THETA

In [60]:
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.sample(10,random_state=random_state).T.style.background_gradient(cmap=colors, axis=None)

doc_title,THE SEVEN RAVENS,THE SALAD,LILY AND THE LION,THE THREE LANGUAGES,IRON HANS,LILY AND THE LION,THE GOLDEN GOOSE,RUMPELSTILTSKIN,THE SEVEN RAVENS,LILY AND THE LION
chunk_id,5,14,13,3,24,17,9,6,0,8
topic_id,,,,,,,,,,
T0,0.008402,0.369411,0.008790,0.383165,0.011848,0.011838,0.010104,0.385375,0.010596,0.012832
T1,0.264445,0.010758,0.008824,0.278311,0.532905,0.157663,0.568522,0.358146,0.109250,0.556575
T2,0.710166,0.010994,0.009011,0.013830,0.011862,0.315833,0.010255,0.235998,0.010699,0.404945
T3,0.008518,0.221984,0.354016,0.013478,0.011843,0.131076,0.010123,0.010212,0.076931,0.012753
T4,0.008469,0.386853,0.619360,0.311215,0.431542,0.383589,0.400996,0.010269,0.792524,0.012895


In [61]:
THETA_vol=THETA.join(LIB)

THETA_vol_agg=THETA_vol.groupby('volume').mean()
# Got some help from a peer but decided to take the top volume for each topic so I could make comparisions in my TCM
top_volume=THETA_vol_agg.idxmax()
topic_ideas = {
    'T0':'T0: animals',
    'T1': 'T1: family',
    'T2': 'T2: jobs',
    'T3': 'T3: princess tales',
    'T4': 'T4: male-focused'
}
THETA_vol_agg.rename(columns=topic_ideas, inplace=True)
THETA_vol_agg.style.background_gradient(cmap=colors, axis=None)


,T0: animals,T1: family,T2: jobs,T3: princess tales,T4: male-focused,n_tokens,n_paras
volume,,,,,,,
1,0.158186,0.299587,0.180978,0.160304,0.200945,2270.744227,19.413854
2,0.085384,0.265408,0.199166,0.176874,0.273168,2490.438914,16.257919


## PHI

In [62]:
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.T.sample(10,random_state=random_state).T.style.background_gradient(cmap=colors, axis=None)

term_str,princes,calf,cabin,fast,hours,hurry,friends,huntsman,meantime,cock
topic_id,,,,,,,,,,
T0,0.202062,0.201044,0.200002,2.199751,0.202663,1.199468,4.200633,0.200585,0.220635,13.207793
T1,4.953353,6.317614,0.200001,0.200295,3.675232,1.234863,0.202267,0.201429,1.635684,5.185661
T2,0.200979,0.200003,6.199310,3.199667,0.203766,2.201169,3.188530,0.204469,0.209585,0.202249
T3,0.216484,11.081337,0.200169,0.200003,0.208327,0.201346,0.201959,0.201019,0.221740,0.202677
T4,7.427121,0.200002,0.200518,0.200284,2.710013,2.163155,0.206611,41.192498,9.712357,0.201619


## Get Top Terms By Topic

In [63]:
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS['doc_weight_mean']=THETA.mean()
TOPICS


,top_terms,doc_weight_mean
topic_id,,
T0,cat fox house door dog,0.137664
T1,man king wife day mother,0.289952
T2,king tailor queen cook time,0.186105
T3,princess horse castle morning peasant,0.164975
T4,father king son bird heart,0.221303


## LDA + PCA

In [64]:
pca_engine = PCA(n_components=4)
TCM = pd.DataFrame(pca_engine.fit_transform(THETA.T),index=THETA.T.index)
TCM.columns = ['PC{}'.format(i) for i in TCM.columns]
TCM['doc_mean_weight']= TCM.mean(axis=1)
TCM['top_volume'] = top_volume
TCM.style.background_gradient(cmap=colors, axis=None)

,PC0,PC1,PC2,PC3,doc_mean_weight,top_volume
topic_id,,,,,,
T0,-2.471059,-2.532599,-2.195866,6.428052,-0.192868,1
T1,10.007763,-0.351966,0.141525,-0.096847,2.425119,1
T2,-2.850886,-3.090341,6.693149,-1.628203,-0.219070,2
T3,-2.583531,-2.109036,-5.167531,-4.830058,-3.672539,2
T4,-2.102288,8.083942,0.528723,0.127057,1.659358,2


In [65]:
def vis_pcs(a, b,DCM,):
    fig =px.scatter(DCM, 
        f"PC{a}", f"PC{b}", 
        color=DCM['top_volume'].astype('category'), 
        size= np.abs(DCM['doc_mean_weight']),
        hover_name=DCM.index,
        text=DCM.index,
        marginal_x='box', 
        height=1000, 
        width=1200)
    return fig

In [66]:
fig=vis_pcs(1,2,TCM)
fig.write_image('TCM.png')
fig.show()

Topics 0 and 3 seems to have similar contexts on average with having a negative PC1 and a negative PC2 values. This makes sense because both topics have at least one animal in the top 5 terms. Topics 1 and 2 are also simialr with the PC1 being negative and PC2 being positive. This make sense because both topic have to do with people and labelsa person can have such as a relation or job. Topic 4 has no neigbor in its quadrant, but is relatively close with topic 1, which makes sense since they both have topic words.

## Save Files to Output

In [67]:
THETA.to_csv(f"{output_dir}/pg2591-THETA.csv", index=True)
PHI.to_csv(f"{output_dir}/pg2591-PHI.csv", index=True)
TOPICS.to_csv(f"{output_dir}/pg2591-TOPICS.csv", index=True)
DTM.to_csv(f"{output_dir}/pg2591-LDA_DTM.csv", index=True)